# Personal Knowledge Decay Predictor

# Notebook 01.5

# Feature Audit & Research Analysis

---

## Objective

The purpose of this notebook is to perform a complete research-level audit of every feature in the ASSISTments dataset.

Instead of directly training machine learning models, we first understand every variable and determine its usefulness for predicting knowledge decay.

The final output of this notebook will be a Feature Audit Report that guides feature engineering, target engineering, and model development.

# Step 2

## Import Libraries

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

# Step 3

## Load Interim Dataset

We load the cleaned interaction dataset created in Notebook 01.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import sys
from pathlib import Path

PROJECT_NAME = "Personal Knowledge Decay Predictor"

matches = [
    p for p in Path("/content/drive/MyDrive").rglob(PROJECT_NAME)
    if p.is_dir()
]

PROJECT_ROOT = matches[0]

sys.path.append(str(PROJECT_ROOT))

from config import *

✅ Project Root : /content/drive/MyDrive/Personal Knowledge Decay Predictor


In [4]:
import pandas as pd
df = pd.read_csv(
    f"{DATA_DIR}/Interim/interactions_clean.csv",
    low_memory=False
)

print(df.shape)

(942816, 82)


# Step 4

## Display Feature List

In [5]:
features = pd.DataFrame({
    "Feature": df.columns
})

features

,Feature
0,studentId
1,MiddleSchoolId
2,InferredGender
3,SY ASSISTments Usage
4,AveKnow
...,...
77,Ln
78,MCAS
79,Enrolled
80,Selective


# Step 5

## Create Feature Audit Table

This table will become the master reference for the project.

In [6]:
feature_audit = pd.DataFrame()

feature_audit["Feature"] = df.columns

feature_audit["Meaning"] = ""

feature_audit["Category"] = ""

feature_audit["Feature Level"] = ""

feature_audit["Keep"] = ""

feature_audit["Leakage Risk"] = ""

feature_audit["Reason"] = ""

feature_audit["Future Engineered Feature"] = ""

feature_audit.head()

,Feature,Meaning,Category,Feature Level,Keep,Leakage Risk,Reason,Future Engineered Feature
0,studentId,,,,,,,
1,MiddleSchoolId,,,,,,,
2,InferredGender,,,,,,,
3,SY ASSISTments Usage,,,,,,,
4,AveKnow,,,,,,,


In [7]:
list(df.columns)

['studentId',
 'MiddleSchoolId',
 'InferredGender',
 'SY ASSISTments Usage',
 'AveKnow',
 'AveCarelessness',
 'AveCorrect',
 'NumActions',
 'AveResBored',
 'AveResEngcon',
 'AveResConf',
 'AveResFrust',
 'AveResOfftask',
 'AveResGaming',
 'action_num',
 'skill',
 'problemId',
 'problemType',
 'assignmentId',
 'assistmentId',
 'startTime',
 'endTime',
 'timeTaken',
 'correct',
 'original',
 'hint',
 'hintCount',
 'hintTotal',
 'scaffold',
 'bottomHint',
 'attemptCount',
 'frIsHelpRequest',
 'frPast5HelpRequest',
 'frPast8HelpRequest',
 'stlHintUsed',
 'past8BottomOut',
 'totalFrPercentPastWrong',
 'totalFrPastWrongCount',
 'frPast5WrongCount',
 'frPast8WrongCount',
 'totalFrTimeOnSkill',
 'timeSinceSkill',
 'frWorkingInSchool',
 'totalFrAttempted',
 'totalFrSkillOpportunities',
 'responseIsFillIn',
 'responseIsChosen',
 'endsWithScaffolding',
 'endsWithAutoScaffolding',
 'frTimeTakenOnScaffolding',
 'frTotalSkillOpportunitiesScaffolding',
 'totalFrSkillOpportunitiesByScaffolding',
 'frI

🟦 Step 6 — Create Feature Groups

In [8]:
student_features = [
    "studentId",
    "MiddleSchoolId",
    "InferredGender",
    "SY ASSISTments Usage",
    "Enrolled",
    "Selective",
    "isSTEM"
]

knowledge_features = [
    "AveKnow",
    "AveCorrect",
    "AveCarelessness"
]

emotion_features = [
    "AveResBored",
    "AveResEngcon",
    "AveResConf",
    "AveResFrust",
    "AveResOfftask",
    "AveResGaming",

    "confidence(BORED)",
    "confidence(CONCENTRATING)",
    "confidence(CONFUSED)",
    "confidence(FRUSTRATED)",
    "confidence(OFF TASK)",
    "confidence(GAMING)",

    "RES_BORED",
    "RES_CONCENTRATING",
    "RES_CONFUSED",
    "RES_FRUSTRATED",
    "RES_OFFTASK",
    "RES_GAMING"
]

problem_features = [
    "problemId",
    "problemType",
    "assignmentId",
    "assistmentId",
    "original"
]

temporal_features = [
    "startTime",
    "endTime",
    "timeTaken",
    "timeSinceSkill"
]

skill_features = [
    "skill",
    "sumTimePerSkill",
    "totalFrTimeOnSkill",
    "totalFrSkillOpportunities"
]

behavior_features = [
    "hint",
    "hintCount",
    "hintTotal",
    "bottomHint",
    "scaffold",
    "attemptCount"
]

performance_features = [
    "correct",
    "sumRight",
    "manywrong",
    "consecutiveErrorsInRow"
]

competition_features = [
    "MCAS",
    "Ln",
    "Ln-1"
]

🟦 Step 7 — Automatically Assign Categories

In [9]:
groups = {
    "Student": student_features,
    "Knowledge": knowledge_features,
    "Emotion": emotion_features,
    "Problem": problem_features,
    "Temporal": temporal_features,
    "Skill": skill_features,
    "Behavior": behavior_features,
    "Performance": performance_features,
    "Competition": competition_features
}

In [10]:
feature_audit["Category"] = "Other"

for category, feature_list in groups.items():

    feature_audit.loc[
        feature_audit["Feature"].isin(feature_list),
        "Category"
    ] = category

feature_audit

,Feature,Meaning,Category,Feature Level,Keep,Leakage Risk,Reason,Future Engineered Feature
0,studentId,,Student,,,,,
1,MiddleSchoolId,,Student,,,,,
2,InferredGender,,Student,,,,,
3,SY ASSISTments Usage,,Student,,,,,
4,AveKnow,,Knowledge,,,,,
...,...,...,...,...,...,...,...,...
77,Ln,,Competition,,,,,
78,MCAS,,Competition,,,,,
79,Enrolled,,Student,,,,,
80,Selective,,Student,,,,,


🟦 Step 8 — Check Category Distribution

In [11]:
feature_audit["Category"].value_counts()

,count
Category,
Other,28
Emotion,18
Student,7
Behavior,6
Problem,5
Skill,4
Performance,4
Temporal,4
Knowledge,3


🟦 Step 9 — Research Decision

In [12]:
drop_features = [

    "MCAS",

    "MiddleSchoolId",

    "Selective",

    "Enrolled",

    "isSTEM"

]

In [13]:
review_features = [

    "AveKnow",

    "AveCorrect",

    "Ln",

    "Ln-1"

]

In [14]:
feature_audit["Keep"] = "YES"

feature_audit.loc[
    feature_audit["Feature"].isin(drop_features),
    "Keep"
] = "NO"

feature_audit.loc[
    feature_audit["Feature"].isin(review_features),
    "Keep"
] = "REVIEW"

🟦 Step 10 — Leakage Column

In [15]:
feature_audit["Leakage Risk"] = "Low"

feature_audit.loc[
    feature_audit["Feature"].isin(review_features),
    "Leakage Risk"
] = "High"

🟦 Step 11 — Feature Level (Research Novelty)

In [16]:
raw_features = [

    "studentId",

    "skill",

    "correct",

    "timeTaken",

    "hintCount",

    "attemptCount",

    "startTime",

    "endTime"

]

ASSISTments engineered features

In [17]:
existing_engineered = [

    "AveKnow",

    "AveCorrect",

    "sumRight",

    "manywrong",

    "Prev5count",

    "timeSinceSkill",

    "sumTimePerSkill"

]

Assign levels.

In [18]:
feature_audit["Feature Level"] = "L2"

feature_audit.loc[
    feature_audit["Feature"].isin(raw_features),
    "Feature Level"
] = "L1"

feature_audit.loc[
    feature_audit["Feature"].isin(existing_engineered),
    "Feature Level"
] = "L2"

🟦 Step 12 — Save Feature Audit

In [19]:
import os

print(os.getcwd())

/content/drive/.shortcut-targets-by-id/1LWX90DlmGF9QpN_sG10QqlUaqgqBZdLn/Personal Knowledge Decay Predictor


In [20]:
feature_audit.to_excel(
    "Reports/Feature_Audit_Catalog.xlsx",
    index=False
)